# 3 — Risk differentiation : classes homogènes de risque

Ce notebook regroupe les scores d'un segment en **classes homogènes de risque (CHR)** : un petit nombre de classes ordonnées, chacune rassemblant des clients de risque comparable. Les CHR sont l'unité sur laquelle se fera la quantification du risque (chapitre 8) : chaque classe recevra une PD.

Le notebook est **paramétré par segment** : il suffit de changer `CODE` (`"SA"`, `"SB"` ou `"SM"`) pour l'appliquer à un autre segment, à condition que les notebooks 1 et 2 aient été exécutés pour ce segment. Il produit, pour chaque segment, une base au format commun (segment, score, PD du score, CHR) qui sera rassemblée avec celles des autres segments dans le notebook de quantification (échelle maître, chapitre 7.5 et chapitre 8).

**Une bonne mise en classes doit être :**
- **ordonnée** : le taux de défaut augmente d'une classe à la suivante ;
- **stable dans le temps** (7.2) : l'ordre des classes et leur volume se maintiennent d'un mois à l'autre ;
- **homogène à l'intérieur** (7.3) : le risque d'une classe ne varie pas selon d'autres caractéristiques des clients ;
- **hétérogène entre classes** (7.4) : deux classes voisines ont des risques clairement différents ;
- **fidèle au score** (7.6) : le regroupement perd peu de pouvoir discriminant par rapport au score continu.

In [1]:
# Cellule 1 — paramètres et chargement
from pathlib import Path
import polars as pl
from utils import binning, chr as chrm, exploration

pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1); pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_hide_column_data_types(True)

CODE = "SB"                       # segment étudié : "SA", "SB" ou "SM"
ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
VARIABLES_EXOGENES = ["ANCIENNETE_modif", "AGE_PP_EN_MOIS"]   # variables non retenues par la régression, pour l'homogénéité

base = pl.read_parquet(f"data/processed/base_score_{CODE}.parquet")
modele = pl.read_parquet(f"data/processed/base_modele_{CODE}.parquet")
VARIABLES_EXOGENES = [v for v in VARIABLES_EXOGENES if v in modele.columns]
base = base.join(modele.select(ID, DATE, *VARIABLES_EXOGENES), on=[ID, DATE], how="left")
regles_binning = binning.charger_regles(f"outputs/scorecard/regles_{CODE}.json")

train, test = base.filter(pl.col("echantillon") == "train"), base.filter(pl.col("echantillon") == "test")
print(f"{CODE} : {train.height} lignes train ({train[TARGET].sum():.0f} défauts) | {test.height} lignes test ({test[TARGET].sum():.0f} défauts)")
print(f"score de {base['score'].min():.0f} à {base['score'].max():.0f}, {base['score'].n_unique()} valeurs distinctes")

SB : 88629 lignes train (1091 défauts) | 37996 lignes test (470 défauts)
score de 466 à 710, 219 valeurs distinctes


## 3.1 Construction des classes (7.1)

Trois méthodes sont comparées, toutes estimées sur l'apprentissage :

| Méthode | Principe |
|---|---|
| **Quantiles + fusion** | découpage fin du score en 30 tranches de même effectif, puis fusion des tranches voisines qui ne respectent pas les règles de stabilité |
| **Jenks** (natural breaks) | coupures qui minimisent la variance de la PD à l'intérieur des classes, pour un nombre de classes k fixé ; la qualité se lit dans le GVF (*goodness of variance fit*, proche de 1 = classes homogènes) |
| **Jenks + fusion** | les coupures de Jenks (k = 10 à 15) servent de découpage fin, puis les classes voisines qui ne respectent pas les règles sont fusionnées |

Les **règles de fusion** sont celles du chapitre 4, appliquées cette fois au score : chaque classe doit compter au moins 5 % de la population et 20 défauts, le taux de défaut doit être monotone, et deux classes voisines doivent avoir un écart de risque d'au moins 30 % **sur chacun des mois**. Jenks seul ne garantit pas ces règles ; il est montré à titre de comparaison.

Le critère de choix est la **perte d'information** (7.6) : on compare, sur le test, le Gini du score continu et celui des CHR (chaque client recevant le taux de défaut de sa classe). On retient, parmi les méthodes qui respectent les règles, celle qui perd le moins de Gini.

In [2]:
# Cellule 2 — comparaison des méthodes de construction
PARAMS_CHR = dict(part_min=0.05, defauts_min=20, ecart_min=0.3)
candidats, lignes = {}, []

candidats["quantiles + fusion"] = chrm.chr_par_regroupement(train, "score", TARGET, DATE, n_fin=30, **PARAMS_CHR)
for k in range(5, 13):
    regle_k, gvf = chrm.chr_jenks(train, "score", "pd_score", k)
    candidats[f"jenks k={k}"] = regle_k | {"gvf": gvf}
for k in (10, 12, 15):
    regle_k, _ = chrm.chr_jenks(train, "score", "pd_score", k)
    candidats[f"jenks k={k} + fusion"] = chrm.chr_par_regroupement(train, "score", TARGET, DATE, bornes_fines=regle_k["bornes"], **PARAMS_CHR)

for nom, regle in candidats.items():
    perte = chrm.perte_information(test, "score", TARGET, regle)
    _, psi = chrm.stabilite_chr(train, "score", TARGET, DATE, regle)
    stab = chrm.stabilite_chr(train, "score", TARGET, DATE, regle)[0]["ecart_min_suivante"].drop_nulls().drop_nans()
    lignes.append({"methode": nom, "n_classes": len(regle["bornes"]) + 1, "regles_respectees": "fusion" in nom,
                   "gvf": regle.get("gvf"), "ecart_min_mois": stab.min() if stab.len() else None, "psi_max": psi,
                   "gini_score_test": perte["gini_score"], "gini_chr_test": perte["gini_chr"], "perte_points": perte["perte_points"]})
comparaison_chr = pl.DataFrame(lignes)
comparaison_chr

methode,n_classes,regles_respectees,gvf,ecart_min_mois,psi_max,gini_score_test,gini_chr_test,perte_points
"""quantiles + fusion""",5,true,null,0.733751,0.000832,0.815409,0.789995,2.54137
"""jenks k=5""",5,false,0.93495,0.156202,0.000318,0.815409,0.626601,18.880833
"""jenks k=6""",6,false,0.955458,0.156202,0.000689,0.815409,0.719319,9.609003
"""jenks k=7""",7,false,0.967347,0.002734,0.000683,0.815409,0.719294,9.611486
"""jenks k=8""",8,false,0.975708,-0.341609,0.000836,0.815409,0.721877,9.353146
"""jenks k=9""",9,false,0.981325,-0.116412,0.000886,0.815409,0.764938,5.047099
"""jenks k=10""",10,false,0.984741,-0.607143,0.000932,0.815409,0.765013,5.039553
"""jenks k=11""",11,false,0.988143,-0.607143,0.001076,0.815409,0.791698,2.371139
"""jenks k=12""",12,false,0.989668,-0.607143,0.001081,0.815409,0.79839,1.701861
"""jenks k=10 + fusion""",3,true,null,3.198203,0.00024,0.815409,0.74116,7.4249


In [3]:
# Cellule 3 — CHR retenues : méthode respectant les règles avec la plus faible perte de Gini
CHOIX = comparaison_chr.filter(pl.col("regles_respectees")).sort("perte_points")["methode"][0]
regle_chr = {k: v for k, v in candidats[CHOIX].items() if k != "gvf"}
print("méthode retenue :", CHOIX, "| bornes de score :", regle_chr["bornes"])

table_train = chrm.table_chr(train, "score", TARGET, regle_chr, "pd_score")
table_test = chrm.table_chr(test, "score", TARGET, regle_chr, "pd_score")
print("Apprentissage"); print(table_train)
print("Test"); print(table_test)

méthode retenue : quantiles + fusion | bornes de score : [571.0, 602.0, 621.0, 668.0]
Apprentissage
shape: (5, 9)
┌─────┬──────────────────┬───────┬─────────────────┬──────────┬─────────────┬──────────┬──────────┬────────────────────┐
│ chr ┆ intervalle_score ┆ n     ┆ part_population ┆ n_defaut ┆ taux_defaut ┆ ic_bas   ┆ ic_haut  ┆ pd_predite_moyenne │
╞═════╪══════════════════╪═══════╪═════════════════╪══════════╪═════════════╪══════════╪══════════╪════════════════════╡
│ 1   ┆ ]668 ; +inf]     ┆ 47221 ┆ 0.532794        ┆ 23.0     ┆ 0.000487    ┆ 0.000237 ┆ 0.000818 ┆ 0.000881           │
│ 2   ┆ ]621 ; 668]      ┆ 23669 ┆ 0.267057        ┆ 97.0     ┆ 0.004098    ┆ 0.003361 ┆ 0.004996 ┆ 0.004425           │
│ 3   ┆ ]602 ; 621]      ┆ 5837  ┆ 0.065859        ┆ 93.0     ┆ 0.015933    ┆ 0.013024 ┆ 0.019478 ┆ 0.012901           │
│ 4   ┆ ]571 ; 602]      ┆ 5725  ┆ 0.064595        ┆ 187.0    ┆ 0.032664    ┆ 0.028363 ┆ 0.037591 ┆ 0.029653           │
│ 5   ┆ ]-inf ; 571]     ┆ 6177  ┆ 0.06

**Résultats.** La méthode retenue est le découpage en quantiles suivi de la fusion des classes voisines : elle produit **5 CHR**, avec un Gini de 0,790 sur le test contre 0,815 pour le score continu, soit une perte limitée à 2,5 points. Jenks seul perd un peu moins d'information avec 11 ou 12 classes, mais ses classes se croisent certains mois (écart minimal négatif) et ne sont donc pas exploitables ; combiné à la fusion, il aboutit à 3 ou 4 classes et perd davantage.

Le taux de défaut croît fortement d'une classe à l'autre, de 0,05 % (classe 1, 53 % de la population) à 11 % (classe 5, 7 % de la population), et les taux observés sur le test sont proches de ceux de l'apprentissage. La classe 1 rassemble plus de la moitié du segment : elle regroupe notamment les clients au score maximal, que le score ne permet pas de distinguer entre eux.


## 3.2 Stabilité dans le temps (7.2)

Par construction, les classes sont ordonnées sur l'ensemble de l'apprentissage. Mais **mois par mois**, deux classes voisines peuvent se croiser (la classe 3 moins risquée que la classe 2 en avril, par exemple). Le tableau donne le taux de défaut de chaque classe pour chaque mois et, dans `ecart_min_suivante`, le plus petit écart relatif avec la classe voisine sur l'ensemble des mois : une valeur négative signale un croisement. La stabilité en **volume** est mesurée par le PSI de la répartition des classes d'un mois à l'autre (< 0,1 : stable). Le contrôle est fait sur l'apprentissage et sur le test.

In [4]:
# Cellule 4 — stabilité en risque (croisements par mois) et en volume (PSI)
for nom, d in (("Apprentissage", train), ("Test", test)):
    stab, psi = chrm.stabilite_chr(d, "score", TARGET, DATE, regle_chr)
    print(f"{nom} — PSI maximal entre mois : {psi:.4f}"); print(stab)

Apprentissage — PSI maximal entre mois : 0.0008
shape: (5, 6)
┌──────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬────────────────────┐
│ classe       ┆ taux_2021-01-01 ┆ taux_2021-02-01 ┆ taux_2021-03-01 ┆ taux_2021-04-01 ┆ ecart_min_suivante │
╞══════════════╪═════════════════╪═════════════════╪═════════════════╪═════════════════╪════════════════════╡
│ ]-inf ; 571] ┆ 0.100499        ┆ 0.114248        ┆ 0.111692        ┆ 0.121631        ┆ 1.829747           │
│ ]571 ; 602]  ┆ 0.035515        ┆ 0.033657        ┆ 0.031975        ┆ 0.029477        ┆ 0.733751           │
│ ]602 ; 621]  ┆ 0.014644        ┆ 0.01817         ┆ 0.018443        ┆ 0.012388        ┆ 2.220359           │
│ ]621 ; 668]  ┆ 0.004198        ┆ 0.00434         ┆ 0.003997        ┆ 0.003847        ┆ 5.526416           │
│ ]668 ; +inf] ┆ 0.000426        ┆ 0.000421        ┆ 0.000512        ┆ 0.000589        ┆ NaN                │
└──────────────┴─────────────────┴─────────────────┴──────

**Résultats.** Aucun croisement de classes n'apparaît, ni sur l'apprentissage ni sur le test : sur chacun des quatre mois, chaque classe reste au moins 63 % plus risquée que sa voisine. Le PSI maximal entre mois (0,0008) montre que la répartition des clients entre classes est parfaitement stable.


## 3.3 Hétérogénéité entre classes voisines (7.4)

On vérifie que deux classes voisines ont des risques nettement différents. Avec de grands effectifs, une p-value seule est peu informative (le moindre écart devient significatif) : on rapporte aussi des **mesures d'ampleur**.
- **Rapport des taux de défaut** : la classe la plus risquée doit l'être au moins 1,3 fois plus (règle des 30 %).
- **Log odds ratio** : écart de risque sur l'échelle de la régression logistique, adapté aux événements rares.
- **h de Cohen** : taille d'effet usuelle pour comparer deux proportions. **Limite** : pour des taux de défaut de l'ordre du pour cent, h reste faible (< 0,2) même quand le risque double, car il est construit sur l'échelle arcsin √p où les petites proportions sont tassées. Les seuils de Cohen (0,2 / 0,5 / 0,8) ne s'appliquent donc pas aux événements rares ; h est donné pour information, le rapport et le log odds ratio étant les mesures pertinentes ici.
- **Recouvrement des intervalles de Wilson** : si les intervalles de confiance de deux classes voisines se chevauchent, elles ne sont pas clairement séparées.
- **Test de différence de deux proportions** (unilatéral), en complément.

Le contrôle est fait sur le test, échantillon qui n'a pas servi à construire les classes.

In [5]:
# Cellule 5 — hétérogénéité entre classes voisines (test)
chrm.heterogeneite_chr(table_test)

paire,taux_1,taux_2,rapport,h_cohen,log_odds_ratio,ic_se_recouvrent,p_value
"""1 → 2""",0.000792,0.00395,4.988891,0.069496,1.610379,false,7.5671e-10
"""2 → 3""",0.00395,0.017207,4.356353,0.13733,1.485034,false,1.0325e-13
"""3 → 4""",0.017207,0.037228,2.163537,0.125217,0.792326,false,0.000007
"""4 → 5""",0.037228,0.105083,2.822721,0.271934,1.110788,false,1.9711e-21


**Résultats.** Les classes voisines sont nettement séparées sur le test : le taux de défaut est multiplié par 2,2 à 5 d'une classe à la suivante (log odds ratio de 0,8 à 1,6), les intervalles de confiance de Wilson ne se recouvrent jamais et toutes les différences sont significatives. Le h de Cohen reste compris entre 0,07 et 0,27 malgré ces écarts importants, ce qui illustre sa limite pour des taux de défaut faibles.


## 3.4 Homogénéité à l'intérieur des classes (7.3)

Une classe est homogène si son taux de défaut ne dépend pas d'autres caractéristiques des clients : sinon, la classe mélange des populations de risques différents et une PD unique par classe serait biaisée. On le vérifie avec des variables **extérieures au score** : le mois d'observation et des variables non retenues par la régression (`ANCIENNETE_modif`, `AGE_PP_EN_MOIS`), découpées selon leurs classes du notebook 1. Pour chaque CHR, on mesure l'écart maximal de taux de défaut entre sous-groupes (h de Cohen, avec la même limite que ci-dessus) et on teste l'indépendance (Chi²). Les sous-groupes de moins de 30 dossiers sont ignorés.

In [6]:
# Cellule 6 — homogénéité interne selon le mois et des variables extérieures au score (apprentissage)
homogeneite = [chrm.homogeneite_chr(train, "score", TARGET, regle_chr, DATE)]
for v in VARIABLES_EXOGENES:
    homogeneite.append(chrm.homogeneite_chr(train, "score", TARGET, regle_chr, v, regles_binning.get(v)))
homogeneite = pl.concat([h for h in homogeneite if h.height], how="diagonal_relaxed")
homogeneite

chr,variable,n_sous_groupes,taux_min,taux_max,h_cohen_max,p_chi2
1,"""datdelhis""",4,0.000421,0.000589,0.007522,0.926003
2,"""datdelhis""",4,0.003847,0.00434,0.007725,0.976974
3,"""datdelhis""",4,0.012388,0.018443,0.049383,0.497637
4,"""datdelhis""",4,0.029477,0.035515,0.034092,0.832301
5,"""datdelhis""",4,0.100499,0.121631,0.067322,0.30424
1,"""ANCIENNETE_modif""",4,0.0,0.000679,0.052129,0.500439
2,"""ANCIENNETE_modif""",4,0.003199,0.004508,0.021211,0.770301
3,"""ANCIENNETE_modif""",4,0.012528,0.032328,0.137237,0.02191
4,"""ANCIENNETE_modif""",4,0.02193,0.042791,0.119459,0.116233
5,"""ANCIENNETE_modif""",4,0.06422,0.121408,0.199382,0.002251


**Résultats.** Les classes sont homogènes selon le mois d'observation (p > 0,3 pour toutes les classes). En revanche, dans la classe la plus risquée (classe 5), le taux de défaut varie selon l'ancienneté de la relation (de 6,4 % à 12,1 %, p = 0,002) et selon l'âge (de 6,1 % à 12,3 %, p = 0,0003) ; un effet plus faible apparaît dans la classe 3. Ces deux variables ont été écartées par la sélection de la régression, leur effet n'étant pas significatif une fois les autres variables présentes : leur influence est concentrée sur les clients les plus risqués (interaction que la régression additive ne capte pas). Cette hétérogénéité modérée est documentée comme une limite. Elle ne conduit pas à découper la classe 5 selon ces variables, les CHR devant rester définies par le score, et l'âge est en outre un critère sensible au regard de la non-discrimination.


## 3.5 Export

On sauvegarde la règle de découpage du score en CHR et une base au **format commun à tous les segments** (identifiant, date, cible, échantillon, segment, score, PD du score, numéro de CHR). Le notebook de quantification rassemblera les bases des segments disponibles pour construire l'échelle maître et calibrer les PD.

In [7]:
# Cellule 7 — export : règle des CHR, base au format commun, tableaux de contrôle
binning.sauver_regles({"chr": regle_chr}, f"outputs/chr/regle_chr_{CODE}.json")

numeros = dict(zip(table_train["intervalle_score"], table_train["chr"]))
base_chr = base.select(ID, DATE, TARGET, "echantillon", "segment", "score", "pd_score").with_columns(
    chr=binning.appliquer_regle(base, "score", regle_chr)["classe"].replace_strict(numeros, return_dtype=pl.Int64))
base_chr.write_parquet(f"data/processed/base_chr_{CODE}.parquet")

exploration.exporter_excel({
    f"{CODE}_Comparaison": comparaison_chr,
    f"{CODE}_CHR_train": table_train, f"{CODE}_CHR_test": table_test,
    f"{CODE}_Heterogeneite": chrm.heterogeneite_chr(table_test),
    f"{CODE}_Homogeneite": homogeneite,
}, f"outputs/chr/chr_{CODE}.xlsx")
print(base_chr.group_by("echantillon", "chr").agg(pl.len(), pl.col(TARGET).mean()).sort("echantillon", "chr"))

shape: (10, 4)
┌─────────────┬─────┬───────┬─────────────┐
│ echantillon ┆ chr ┆ len   ┆ ddefaut_ndb │
╞═════════════╪═════╪═══════╪═════════════╡
│ test        ┆ 1   ┆ 20209 ┆ 0.000792    │
│ test        ┆ 2   ┆ 10127 ┆ 0.00395     │
│ test        ┆ 3   ┆ 2499  ┆ 0.017207    │
│ test        ┆ 4   ┆ 2525  ┆ 0.037228    │
│ test        ┆ 5   ┆ 2636  ┆ 0.105083    │
│ train       ┆ 1   ┆ 47221 ┆ 0.000487    │
│ train       ┆ 2   ┆ 23669 ┆ 0.004098    │
│ train       ┆ 3   ┆ 5837  ┆ 0.015933    │
│ train       ┆ 4   ┆ 5725  ┆ 0.032664    │
│ train       ┆ 5   ┆ 6177  ┆ 0.111867    │
└─────────────┴─────┴───────┴─────────────┘
